In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
source = Path("../data_raw/Fraility-Grip-Strength-rawdata.csv")
df_raw = pd.read_csv(source)
print(f"Dataset loaded from: {source}")
df_raw["Height "].tolist()

Dataset loaded from: ../data_raw/Fraility-Grip-Strength-rawdata.csv


['65.8\xa0',
 '71.5\xa0',
 '69.4\xa0',
 '68.2\xa0',
 '67.8\xa0',
 '68.7\xa0',
 '69.8\xa0',
 '70.1\xa0',
 '67.9\xa0',
 '66.8\xa0']

In [3]:
print(df_raw.columns.tolist())
df_raw.columns = df_raw.columns.str.strip().str.replace("\xa0", "", regex=False)
for col in df_raw.columns:
    df_raw[col] = df_raw[col].astype(str).str.strip().str.replace("\xa0", "", regex=False)
print(df_raw.columns.tolist())

['Height ', 'Weight ', 'Age\xa0', 'Grip strength\xa0', 'Frailty\xa0']
['Height', 'Weight', 'Age', 'Grip strength', 'Frailty']


In [4]:
df_raw.head(7)
dataset_shape = df_raw.shape
print(f"The dataset has {dataset_shape[0]} rows, and has {dataset_shape[1]} columns.")
df_raw.info()

The dataset has 10 rows, and has 5 columns.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Height         10 non-null     object
 1   Weight         10 non-null     object
 2   Age            10 non-null     object
 3   Grip strength  10 non-null     object
 4   Frailty        10 non-null     object
dtypes: object(5)
memory usage: 532.0+ bytes


In [6]:
IN_TO_M = 0.0254
LB_TO_KG = 0.45359237

df_raw["Height"] = pd.to_numeric(df_raw["Height"], errors="coerce")
df_raw["Weight"] = pd.to_numeric(df_raw["Weight"], errors="coerce")
df_raw["Age"] = pd.to_numeric(df_raw["Age"], errors="coerce")
df_raw["Grip strength"] = pd.to_numeric(df_raw["Grip strength"], errors="coerce")
df_raw["Frailty"] = df_raw["Frailty"].map({'Y': True, 'N': False})

df_raw = df_raw.rename(columns={"Height": "Height_m", "Weight": "Weight_kg"})
df_raw["Height_m"] *= IN_TO_M
df_raw["Weight_kg"] *= LB_TO_KG
df_raw.head(7)

,Height_m,Weight_kg,Age,Grip strength,Frailty
0,1.67132,50.802345,30,30,False
1,1.81610,61.688562,19,31,False
2,1.76276,69.399633,45,29,False
3,1.73228,64.410117,22,28,True
4,1.72212,65.317301,29,24,True
5,1.74498,55.791862,50,26,False
6,1.77292,63.956524,51,22,True


In [7]:
df_raw["BMI"] = (df_raw["Weight_kg"] / (df_raw["Height_m"]**2)).round(2)
df_raw["Age Group (categorical)"] = None

df_raw.loc[df_raw["Age"] < 30, "Age Group (categorical)"] = "<30"
df_raw.loc[(df_raw["Age"] > 29) & (df_raw["Age"] < 46), "Age Group (categorical)"] = "30-45"
df_raw.loc[(df_raw["Age"] > 45) & (df_raw["Age"] < 61), "Age Group (categorical)"] = "45-60"
df_raw.loc[df_raw["Age"] > 60, "Age Group (categorical)"] = ">60"


In [8]:
df_raw.head(7)


,Height_m,Weight_kg,Age,Grip strength,Frailty,BMI,Age Group (categorical)
0,1.67132,50.802345,30,30,False,18.19,30-45
1,1.81610,61.688562,19,31,False,18.70,<30
2,1.76276,69.399633,45,29,False,22.33,30-45
3,1.73228,64.410117,22,28,True,21.46,<30
4,1.72212,65.317301,29,24,True,22.02,<30
5,1.74498,55.791862,50,26,False,18.32,45-60
6,1.77292,63.956524,51,22,True,20.35,45-60


In [10]:
numeric_columns = df_raw.select_dtypes(include="number").columns.tolist()

summary_stats_df = df_raw[numeric_columns].describe()
summary_stats_df.to_markdown('../reports/findings.md')

correlation = df_raw['Grip strength'].corr(df_raw['Frailty'])

with open('../reports/findings.md', 'a') as f:
    f.write(f'\n\nThe Correlation Coefficient between Grip Strength and fraility is: {correlation:.3f}')